# m01 · Introduction, principles of propulsion and engine classification: verification calculations

This notebook recomputes in Python the main examples and formulas of module m01 (Slide Ch.1; El-Sayed Ch.2; Farokhi Ch.3, Appendix A). Every section has three parts: the question the method answers → code → interpretation of the real printed numbers. Run top to bottom (needs numpy, scipy, sympy, matplotlib).

## 1. ISA and checking the numbers of slide p.48

**Question:** slide Ch.1 p.48 labels the tropopause at −56 °C and 264 hPa. Which altitude does that correspond to in the ISA, and how far is the ISA from Farokhi Appendix A?

In [1]:
import math
def isa(z):
    """ISA 0-20 km: troposphere T = 288.15 - 0.0065 z, isothermal 216.65 K above 11 km."""
    T0, p0, L, Rr, g = 288.15, 101325.0, 0.0065, 287.053, 9.80665
    if z <= 11000:
        T = T0 - L*z; p = p0*(T/T0)**(g/(Rr*L))
    else:
        T = 216.65; p = p0*(T/T0)**(g/(Rr*L))*math.exp(-g*(z-11000)/(Rr*T))
    return T, p, p/(Rr*T)
# Farokhi Appendix A (SI) typed from the page image: alt km -> (T K, p Pa)
farokhi = {0:(288.1,1.013e5), 2:(275.2,7.950e4), 4:(262.2,6.166e4), 6:(249.2,4.722e4), 8:(236.2,3.565e4), 10:(223.3,2.650e4), 12:(216.6,1.940e4), 20:(216.6,5.529e3)}
print("z km |  T ISA   T Farokhi |  p ISA (Pa)  p Farokhi (Pa) | dp %")
for z,(Tf,pf) in farokhi.items():
    T,p,rho = isa(z*1000.)
    print(f"{z:4d} | {T:7.2f} {Tf:9.1f} | {p:11.1f} {pf:14.1f} | {100*(p-pf)/pf:+.2f}")
for z in (10000., 11000., 25000.):
    T,p,rho = isa(z); print(f"z = {z/1000:.0f} km: T = {T-273.15:.2f} C, p = {p/100:.2f} hPa, rho = {rho:.4f} kg/m3")

z km |  T ISA   T Farokhi |  p ISA (Pa)  p Farokhi (Pa) | dp %
   0 |  288.15     288.1 |    101325.0       101300.0 | +0.02
   2 |  275.15     275.2 |     79495.2        79500.0 | -0.01
   4 |  262.15     262.2 |     61640.2        61660.0 | -0.03
   6 |  249.15     249.2 |     47181.0        47220.0 | -0.08
   8 |  236.15     236.2 |     35599.8        35650.0 | -0.14
  10 |  223.15     223.3 |     26436.3        26500.0 | -0.24
  12 |  216.65     216.6 |     19330.4        19400.0 | -0.36
  20 |  216.65     216.6 |      5474.9         5529.0 | -0.98
z = 10 km: T = -50.00 C, p = 264.36 hPa, rho = 0.4127 kg/m3
z = 11 km: T = -56.50 C, p = 226.32 hPa, rho = 0.3639 kg/m3
z = 25 km: T = -56.50 C, p = 24.89 hPa, rho = 0.0400 kg/m3


**Reading the output:** the ISA gives p(10 km) = 264.4 hPa and p(11 km) = 226.3 hPa; −56 °C (-56.50 °C) belongs to 11 km. So the "264 hPa" on the slide is the pressure at 10 km paired with the temperature of 11 km; this is a slide error, not an OCR error. ISA differs little from Farokhi's table (≤ 0.4 % up to 12 km). The 26 hPa mark at ~25 km: ISA 24.89 hPa (rounded).

## 2. Four forces in level flight

**Question:** an aircraft of m = 60,000 kg in level flight with C_L = 0.5, C_D = 0.03. What thrust is needed and how fast must it fly at 10 km (ISA) so that F_z = P, if S = 122 m²?

In [2]:
import math
G0 = 9.80665
m, CL, CD, S_ = 60000., 0.5, 0.03, 122.
P = m*G0
Fx = CD/CL*P                      # T_u = F_x, from F_z = P
print(f"weight P = {P/1e3:.1f} kN, required thrust T_u = {Fx/1e3:.2f} kN = {100*Fx/P:.2f} % of weight")
for z in (0., 10000.):
    T,p,rho = isa(z)
    V = math.sqrt(2*P/(rho*S_*CL))      # from F_z = CL*V^2*rho/2*S = P
    Fz = CL*V**2*rho/2*S_
    print(f"z = {z/1000:4.0f} km: rho = {rho:.4f}  V = {V:.1f} m/s = {3.6*V:.0f} km/h  (check F_z = {Fz/1e3:.1f} kN)")

weight P = 588.4 kN, required thrust T_u = 35.30 kN = 6.00 % of weight
z =    0 km: rho = 1.2250  V = 125.5 m/s = 452 km/h  (check F_z = 588.4 kN)
z =   10 km: rho = 0.4127  V = 216.2 m/s = 778 km/h  (check F_z = 588.4 kN)


**Reading the output:** the required thrust is only 35.30 kN, about 6.0 % of the weight (= C_D/C_L): the engine does not lift the aircraft. At 10 km the density is about a third of the sea-level one, so the speed needed to keep the lift rises by √(ρ0/ρ) ≈ 1.7.

## 3. The thrust equation: El-Sayed Examples 2.1 and 2.2

**Question:** (a) with ṁa = 30 kg/s, ṁf = 1 kg/s, ue = 610 m/s and a thrust power of 1.12×10⁶ W, what is the flight speed u? (b) the forward thrust and the thrust with the exhaust turned 90° of an idling engine during landing?

In [3]:
import sympy as sp
u = sp.symbols('u', positive=True)
roots = sp.solve(sp.Eq((30+1)*610*u - 30*u**2, 1.12e6), u)      # thrust power = T*u = (ma+mf) ue u - ma u^2
print("Example 2.1 roots u (m/s):", [round(float(r),2) for r in roots])
# Example 2.2 (British units): ma = 100 lbm/s, U = 100 mph, Ue = 450 ft/s, gc = 32.2
U = 100*1.4667; gc = 32.2
Tf = 100/gc*(450-U); Tr = 100/gc*(0-U)
print(f"forward thrust = {Tf:.0f} lbf ; with exhaust turned 90 deg = {Tr:.1f} lbf (negative = braking)")

Example 2.1 roots u (m/s): [66.18, 564.16]
forward thrust = 942 lbf ; with exhaust turned 90 deg = -455.5 lbf (negative = braking)


**Reading the output:** the quadratic has two roots 564.16 and 66.18 m/s, matching the book (564.15 and 66.17). The larger root is the jet's flight state. Forward thrust is 942 lbf; with the jet turned 90° the horizontal component is zero so the force is only −ṁa·U = -455.5 lbf, a braking force: the mechanism of thrust-reverse braking.

## 4. Three propulsive-efficiency expressions (El-Sayed Example 2.5)

**Question:** with a choked nozzle (Pe = 0.67 bar > Pa = 0.265 bar), which expression of η_p is still usable?

In [4]:
import math
gam, Rg, Ta, Ma = 1.4, 287., 223.3, 0.8
ue, pe, pa, f, Ae_over_m = 570., 0.67e5, 0.265e5, 0.02, 0.006
u = Ma*math.sqrt(gam*Rg*Ta)
st = ((1+f)*ue - u) + (pe-pa)*Ae_over_m           # specific thrust, N s/kg
e1 = 2*st*u/(2*st*u + (1+f)*(ue-u)**2)            # expression 1: thrust power / (thrust power + wake loss)
e2 = 2*st*u/((1+f)*ue**2 - u**2)                  # expression 2: / kinetic energy added
e3 = 2*u/(u+ue)                                   # expression 3: Pe = Pa and f = 0 only
print(f"u = {u:.1f} m/s, specific thrust = {st:.2f} N s/kg")
print(f"eta_p: expr1 = {e1:.4f}   expr2 = {e2:.4f}   expr3 = {e3:.4f}")
# effective (fully expanded) jet velocity ue' = ue + (pe-pa) Ae / me ; then both expressions are evaluated with it
ue_eff = ue + (pe-pa)*Ae_over_m/(1+f)
e1b = 2*st*u/(2*st*u + (1+f)*(ue_eff-u)**2)
e2b = 2*st*u/((1+f)*ue_eff**2 - u**2)
print(f"effective jet velocity {ue_eff:.1f} m/s: expr1 = {e1b:.4f}, expr2 = {e2b:.4f}")

u = 239.6 m/s, specific thrust = 584.77 N s/kg
eta_p: expr1 = 0.7157   expr2 = 1.0229   expr3 = 0.5919
effective jet velocity 808.2 m/s: expr1 = 0.4594, expr2 = 0.4603


**Reading the output:** expression 1 gives 71.57 % (book 71.56 %); expression 2 gives 102.3 %, above 100 % and so meaningless for a choked nozzle; expression 3 is only a rough 59.19 %. If both expressions are evaluated with the effective jet velocity u_e′ (full expansion; last line) they nearly agree (they differ by an f term), but the value ≈ 0.46 is far below the 71.56 % of the book, because the unexpanded pressure energy of the jet is then counted as wake loss. Conclusion: state which expression and which assumption (actual u_e or u_e′) before comparing η_p values.

## 5. CF6 turbofan: three efficiencies in a chain (El-Sayed Example 2.7)

**Question:** from a thrust of 24 kN (fan 75 %), BPR = 5, ṁa = 125 kg/s, ṁf = 0.75 kg/s, Ma 0.8 at 10 km: what are the jet speeds, η_p, η_th, η_o, and do they match the numbers printed in the book?

In [5]:
import math
from scipy.optimize import brentq
T, ma, beta, mf, M, Ta, QR = 24000., 125., 5., 0.75, 0.8, 223.2, 42.8e6
U = M*math.sqrt(1.4*287*Ta)
mc, mh = ma*beta/(1+beta), ma/(1+beta); f = mf/mh
Tfan, Tcore = 0.75*T, 0.25*T
uec = brentq(lambda ue: mc*(ue-U) - Tfan, 1, 3000)                  # solve instead of rearranging
ueh = brentq(lambda ue: mh*((1+f)*ue-U) - Tcore, 1, 3000)
dKE = 0.5*(mh*((1+f)*ueh**2 - U**2) + mc*(uec**2 - U**2))          # power imparted to the airflow
eta_p, eta_th = T*U/dKE, dKE/(mf*QR)
print(f"U = {U:.1f} m/s; mc = {mc:.1f}, mh = {mh:.1f} kg/s, f = {f:.4f}")
print(f"uec = {uec:.1f} m/s (book 413), ueh = {ueh:.1f} m/s (book 510)")
print(f"dKE = {dKE/1e6:.3f} MW; eta_p = {eta_p:.4f} (book 0.712), eta_th = {eta_th:.4f} (book 0.2546), eta_o = {eta_p*eta_th:.4f} (book 0.1793)")
print(f"check eta_o = T U /(mf QR) = {T*U/(mf*QR):.4f};   specific thrust = {T/ma:.1f} N s/kg, TSFC = {mf/T*1e6:.2f} mg/(N s)")

U = 239.6 m/s; mc = 104.2, mh = 20.8 kg/s, f = 0.0360
uec = 412.4 m/s (book 413), ueh = 509.2 m/s (book 510)
dKE = 8.068 MW; eta_p = 0.7126 (book 0.712), eta_th = 0.2513 (book 0.2546), eta_o = 0.1791 (book 0.1793)
check eta_o = T U /(mf QR) = 0.1791;   specific thrust = 192.0 N s/kg, TSFC = 31.25 mg/(N s)


**Reading the output:** u_ec = 412.4 m/s and u_eh = 509.2 m/s match 413 and 510; η_p = 0.7126 matches 0.712. But η_th = 0.2513 differs from the printed 0.2546: 16.179×10⁶/(2×0.75×42.8×10⁶) = 0.2520 (exactly, with unrounded speeds: 0.2513), and the printed η_o = 0.1793 matches only 0.712×0.252. This is an arithmetic slip of the book (listed in the source errors). η_o = 17.91 % also equals F·U/(ṁf·Q).

## 6. Gross, net thrust and ram drag (Farokhi Problems 3.5 and 3.11)

**Question:** how much of the gross thrust is lost to ram drag at Mach 2, and how does that compare with a slow turbojet (3.11)?

In [6]:
import math
def engine(m, f, V9, V0, QR):
    mf = f*m; Dram = m*V0; Fg = (m+mf)*V9; Fn = Fg - Dram
    dKE = 0.5*((m+mf)*V9**2 - m*V0**2)
    return dict(Dram=Dram, Fg=Fg, Fn=Fn, eth=dKE/(mf*QR), ep=Fn*V0/dKE, eo=Fn*V0/(mf*QR))
V0a = 2*math.sqrt(1.4*287*223.15)
a = engine(25, 0.03, 1050, V0a, 42.8e6)          # Farokhi 3.5 (Mach 2)
b = engine(100, 0.025, 600, 300, 42.0e6)         # Farokhi 3.11 (V0 = 300 m/s)
for name, r in (("3.5  Mach 2", a), ("3.11 300 m/s", b)):
    print(f"{name}: Dram = {r['Dram']/1e3:.2f} kN, Fg = {r['Fg']/1e3:.2f} kN, Fn = {r['Fn']/1e3:.2f} kN, ram share = {100*r['Dram']/r['Fg']:.1f} %,")
    print(f"      eta_th = {r['eth']:.4f}, eta_p = {r['ep']:.4f}, eta_o = {r['eo']:.4f}")
print(f"3.5: TSFC = {0.75/a['Fn']*3600:.4f} kg/(h N)  |  3.11: TSFC = {2.5/b['Fn']*1e6:.2f} mg/(s N)")

3.5  Mach 2: Dram = 14.97 kN, Fg = 27.04 kN, Fn = 12.07 kN, ram share = 55.4 %,
      eta_th = 0.3025, eta_p = 0.7440, eta_o = 0.2251
3.11 300 m/s: Dram = 30.00 kN, Fg = 61.50 kN, Fn = 31.50 kN, ram share = 48.8 %,
      eta_th = 0.1329, eta_p = 0.6774, eta_o = 0.0900
3.5: TSFC = 0.2238 kg/(h N)  |  3.11: TSFC = 79.37 mg/(s N)


**Reading the output:** at Mach 2 (V0 = 598.9 m/s) the ram drag 14.97 kN takes 55.4 % of the 27.04 kN gross thrust so the net thrust is only 12.07 kN. At 300 m/s with a 600 m/s jet the ram share is only 49 %, but η_p is much lower when u_e ≫ u. This is why net thrust falls with flight speed.

## 7. Thrust distribution by turbojet component (El-Sayed Example 2.4)

**Question:** what force does each component carry and does the sum equal the thrust of the whole engine?

In [7]:
names = ["compressor", "diffuser", "combustor", "turbine", "jet pipe", "nozzle"]
comps = [(648e3,.117,124),(655e3,.132,112),(641e3,.374,94),(145e3,.310,271),(145e3,.420,196),(41e3,.214,584)]   # gauge P (Pa), A (m2), U (m/s)
m = 69.4
prev = (0.0, 0.0, 0.0); F = []
for (p, A, U) in comps:
    F.append(p*A - prev[0]*prev[1] + m*(U - prev[2]))      # F = Pe Ae - Pi Ai + m (Ue - Ui)
    prev = (p, A, U)
for n, x in zip(names, F): print(f"{n:11s} {x:12.0f} N  {'forward' if x > 0 else 'rearward'}")
print(f"sum of components = {sum(F):.0f} N ; whole engine Pe Ae + m Ue = {41e3*.214 + m*584:.0f} N")

compressor         84422 N  forward
diffuser            9811 N  forward
combustor         152025 N  forward
turbine          -182500 N  rearward
jet pipe           10745 N  forward
nozzle            -25199 N  rearward
sum of components = 49304 N ; whole engine Pe Ae + m Ue = 49304 N


**Reading the output:** the sum of the components = 49304 N = the thrust of the whole engine (book prints 49,304 N). The turbine (-182500 N) and nozzle (-25199 N) pull rearwards; compressor, diffuser, combustor and jet pipe push forward. The printed solution has a few typos (see the source errors).

## 8. Breguet range and the effect of wind (El-Sayed Examples 2.9 and 2.11)

**Question:** what is the range of the Boeing 747 of Example 2.9 and how does a 50 m/s tail/head wind change the fuel for a 4000 km leg?

In [8]:
import math
from scipy.integrate import solve_ivp
# Example 2.9: Breguet S = eta_o QR / g * (L/D) * ln(m1/m2)
S = 0.35*45e6/9.8*17*math.log(1/0.673)
print(f"Example 2.9: S = {S/1e3:.1f} km (book 10,819 km)")
# Example 2.11: integrate dm/ds = -m * TSFC * g / (Vg L/D) over 4000 km, instead of using the closed form
def fuel(Vg, TSFC=0.08/3600, g=9.8, LD=10, s_end=4000e3, m1=50000.):
    sol = solve_ivp(lambda s, m: [-m[0]*TSFC*g/(Vg*LD)], (0, s_end), [m1], rtol=1e-11, atol=1e-9)
    return m1 - sol.y[0, -1]
for name, Vg in (("tail wind 50 m/s (Vg = 300)", 300.), ("no wind (Vg = 250)", 250.), ("head wind 50 m/s (Vg = 200)", 200.)):
    print(f"{name:30s} fuel = {fuel(Vg):8.0f} kg   time = {4000e3/Vg/3600:5.2f} h")

Example 2.9: S = 10819.6 km (book 10,819 km)
tail wind 50 m/s (Vg = 300)    fuel =    12601 kg   time =  3.70 h
no wind (Vg = 250)             fuel =    14711 kg   time =  4.44 h
head wind 50 m/s (Vg = 200)    fuel =    17655 kg   time =  5.56 h


**Reading the output:** S = 10819.6 km matches the book's 10,819 km (ln rounded). The tail wind needs 12601 kg and the head wind 17655 kg (book 12,600 and 17,654): the head wind raises the fuel by ≈ 40 % relative to the tail wind because the ground speed drops (range scales with V_g/TSFC).

## 9. Propulsive efficiency and thrust versus flight speed (plot)

**Question:** what do the shapes of η_p and T versus u/ue (El-Sayed Problem 2.7: ṁa = 50 kg/s, f = 0.015, ue = 500 m/s, Pa = 60 kPa, Pe = 120 kPa, Ae = 0.25 m²) say about takeoff and fast flight?

In [9]:
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
ma, f, ue, pa, pe, Ae = 50., 0.015, 500., 60e3, 120e3, 0.25
x = np.linspace(0, 1, 101); u = x*ue
T = ma*((1+f)*ue - u) + (pe-pa)*Ae                      # thrust
wake = 0.5*ma*(1+f)*(ue-u)**2
eta_p = T*u/(T*u + wake)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4)); a1.plot(x, T/1e3); a1.set_xlabel("u/ue"); a1.set_ylabel("T (kN)"); a1.grid(alpha=.3)
a2.plot(x, eta_p); a2.set_xlabel("u/ue"); a2.set_ylabel("eta_p"); a2.grid(alpha=.3); plt.tight_layout(); plt.savefig("problem27_check.png", dpi=80)
print(f"static thrust (u = 0): {T[0]/1e3:.2f} kN, eta_p = {eta_p[0]:.3f}")
print(f"u/ue = 0.5: T = {T[50]/1e3:.2f} kN, eta_p = {eta_p[50]:.4f};  u/ue = 0.9: T = {T[90]/1e3:.2f} kN, eta_p = {eta_p[90]:.4f}")
print(f"T = 0 would need u = {(ma*(1+f)*ue + (pe-pa)*Ae)/ma:.1f} m/s > ue because of the pressure thrust")

static thrust (u = 0): 40.38 kN, eta_p = 0.000
u/ue = 0.5: T = 27.87 kN, eta_p = 0.8146;  u/ue = 0.9: T = 17.87 kN, eta_p = 0.9922
T = 0 would need u = 807.5 m/s > ue because of the pressure thrust


**Reading the output:** static thrust is 40.38 kN with η_p = 0 (no thrust power yet); as u rises thrust falls linearly to 27.87 kN at u/ue = 0.5 and 17.87 kN at 0.9 while η_p rises. Because of the pressure thrust (Pe &gt; Pa) the thrust does not vanish even at u = ue. This is the basic trade-off: large thrust at low flight speed, high efficiency when the flight speed approaches the jet speed.